In [ ]:
# Cell 1: Install + Clone + Data

import numpy as np
np.int = int; np.float = float

!pip install torch --index-url https://download.pytorch.org/whl/cpu -q
!pip install scipy pandas numba tqdm numpy biopython msgpack scikit-learn PyYAML -q

!rm -rf Enhanced-ECNet
!git clone https://github.com/itozheng-max/Enhanced-ECNet.git
%cd Enhanced-ECNet

!wget -q https://raw.githubusercontent.com/itozheng-max/Enhanced-ECNet/main/ecnet_rrm_data.tar.gz
!tar xzf ecnet_rrm_data.tar.gz

import sys, time, torch
sys.path.insert(0, ".")
print(f"Py {sys.version.split()[0]} | Torch {torch.__version__} | GPU {torch.cuda.is_available()}")

In [ ]:
# Cell 2: Import

from ecnet.ecnet import ECNet
from ecnet.spatial_mask import SpatialMask

FASTA     = "data/RRM.fasta"
SINGLE_TSV = "data/RRM_single.tsv"
DOUBLE_TSV = "data/RRM_double.tsv"
BRAW      = "data/RRM.braw"
DIST_NPY  = "data/distance_matrix.npy"

print("OK")

In [ ]:
# Cell 3: Interactive UI

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

kernel_select = widgets.Dropdown(
    options=['baseline', 'sigmoid', 'hill', 'surprise'],
    value='hill',
    description='Kernel:'
)

task_select = widgets.Dropdown(
    options=[('Single→Single', 'ss'), ('Single→Double', 'sd'), ('Double→Double', 'dd')],
    value='sd',
    description='Task:'
)

d0_slider = widgets.FloatSlider(value=8.0, min=4.0, max=14.0, step=0.5, description='d0 (A)')
gamma_slider = widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1, description='gamma')
n_slider = widgets.IntSlider(value=4, min=1, max=10, step=1, description='n')
eps_slider = widgets.FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description='epsilon')
alpha_slider = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5, description='alpha')

formula_out = widgets.Output()
result_out = widgets.Output()

def show_formula(kernel):
    formulas = {
        'baseline': r'$$ eij_{new} = eij $$',
        'sigmoid': r'$$ W(d) = \frac{1}{1 + e^{\gamma (d - d_0)}} \qquad eij_{new} = \frac{eij}{W + \epsilon} $$',
        'hill': r'$$ W(d) = \frac{1}{1 + (d/d_0)^n} \qquad eij_{new} = \frac{eij}{W + \epsilon} $$',
        'surprise': r'$$ W(d) = \frac{1}{1 + e^{\gamma (d - d_0)}} \qquad eij_{new} = \frac{eij}{W + \epsilon} \cdot \left[1 + \alpha(1-W)\tanh(|eij|_{norm})\right] $$',
    }
    formula_out.clear_output(wait=True)
    with formula_out:
        display(Markdown(f'### Kernel: {kernel}'))
        display(Markdown(formulas[kernel]))

def on_kernel_change(change):
    show_formula(change['new'])
    k = change['new']
    d0_slider.layout.display = 'none' if k == 'baseline' else ''
    gamma_slider.layout.display = '' if k in ('sigmoid', 'surprise') else 'none'
    n_slider.layout.display = '' if k == 'hill' else 'none'
    eps_slider.layout.display = 'none' if k == 'baseline' else ''
    alpha_slider.layout.display = '' if k == 'surprise' else 'none'

kernel_select.observe(on_kernel_change, 'value')
show_formula(kernel_select.value)
on_kernel_change({'new': kernel_select.value})

display(Markdown('---'))
display(Markdown('### Task'))
display(task_select)
display(Markdown('---'))
display(Markdown('### Kernel & Parameters'))
display(kernel_select)
display(formula_out)
display(d0_slider)
display(gamma_slider)
display(n_slider)
display(eps_slider)
display(alpha_slider)

In [ ]:
# Cell 4: Run

def build_mask():
    k = kernel_select.value
    if k == 'baseline':
        return None
    m = {
        'path': DIST_NPY,
        'mode': 'surprise' if k == 'surprise' else 'divide',
        'd0': d0_slider.value,
        'epsilon': eps_slider.value,
    }
    if k == 'sigmoid':
        m['kernel'] = 'sigmoid'
        m['gamma'] = gamma_slider.value
    elif k == 'hill':
        m['kernel'] = 'hill'
        m['n'] = n_slider.value
    elif k == 'surprise':
        m['kernel'] = 'sigmoid'
        m['gamma'] = gamma_slider.value
        m['alpha'] = alpha_slider.value
    return m

def run_one(name, mask):
    ts = task_select.value
    train_tsv = SINGLE_TSV if ts in ('ss', 'sd') else DOUBLE_TSV
    test_tsv = SINGLE_TSV if ts == 'ss' else DOUBLE_TSV
    ecnet = ECNet(
        output_dir=f"./output/{name}",
        train_tsv=train_tsv, test_tsv=test_tsv,
        fasta=FASTA, ccmpred_output=BRAW,
        use_loc_feat=True, use_glob_feat=False,
        split_ratio=[0.9, 0.1] if test_tsv else [0.7, 0.1, 0.2],
        n_ensembles=1, d_embed=20, d_model=64, d_h=64,
        batch_size=64, save_log=False,
        spatial_mask=mask,
    )
    ecnet.train(epochs=200, patience=100, eval_freq=20, log_freq=200)
    return ecnet.test(mode="ensemble")["metric"]["corr"]

result_out.clear_output(wait=True)
with result_out:
    k = kernel_select.value
    ts = task_select.value
    task_names = {'ss': 'Single->Single', 'sd': 'Single->Double', 'dd': 'Double->Double'}
    print(f'Task: {task_names[ts]} | Kernel: {k}')
    if k != 'baseline':
        print(f'  d0={d0_slider.value} eps={eps_slider.value}', end='')
        if k in ('sigmoid', 'surprise'): print(f' gamma={gamma_slider.value}', end='')
        if k == 'hill': print(f' n={n_slider.value}', end='')
        if k == 'surprise': print(f' alpha={alpha_slider.value}', end='')
        print()
    print()

    t0 = time.time()
    b = run_one('Baseline', None)
    print(f'  Baseline -> {b:.6f}  ({time.time()-t0:.0f}s)')

    mask = build_mask()
    if mask is not None:
        t0 = time.time()
        m = run_one(k, mask)
        print(f'  {k:10s} -> {m:.6f}  ({time.time()-t0:.0f}s)')
        print(f'\n  Delta: {m-b:+.6f} ({(m/b-1)*100:+.1f}%)')
    print()

display(result_out)

In [ ]:
# Cell 5: Batch compare (all vs baseline)

def run_one_batch(name, mask):
    ts = task_select.value
    train_tsv = SINGLE_TSV if ts in ('ss', 'sd') else DOUBLE_TSV
    test_tsv = SINGLE_TSV if ts == 'ss' else DOUBLE_TSV
    ecnet = ECNet(
        output_dir=f"./output/batch_{name}",
        train_tsv=train_tsv, test_tsv=test_tsv,
        fasta=FASTA, ccmpred_output=BRAW,
        use_loc_feat=True, use_glob_feat=False,
        split_ratio=[0.9, 0.1] if test_tsv else [0.7, 0.1, 0.2],
        n_ensembles=1, d_embed=20, d_model=64, d_h=64,
        batch_size=64, save_log=False,
        spatial_mask=mask,
    )
    ecnet.train(epochs=200, patience=100, eval_freq=20, log_freq=200)
    return ecnet.test(mode="ensemble")["metric"]["corr"]

print("Running full comparison...\n")

ts = task_select.value
task_names = {'ss': 'Single->Single', 'sd': 'Single->Double', 'dd': 'Double->Double'}
print(f'Task: {task_names[ts]}')
print()

r = {}

t0 = time.time()
r['Baseline'] = run_one_batch('Baseline', None)
print(f'  Baseline         -> {r["Baseline"]:.6f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
r['Sigmoid'] = run_one_batch('Sigmoid', {'path': DIST_NPY, 'mode': 'divide', 'kernel': 'sigmoid', 'd0': 8.0, 'gamma': 1.5, 'epsilon': 0.10})
print(f'  Sigmoid          -> {r["Sigmoid"]:.6f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
r['Hill(n=2)'] = run_one_batch('Hill_n2', {'path': DIST_NPY, 'mode': 'divide', 'kernel': 'hill', 'd0': 8.0, 'n': 2, 'epsilon': 0.10})
print(f'  Hill(n=2)        -> {r["Hill(n=2)"]:.6f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
r['Hill(n=4)'] = run_one_batch('Hill_n4', {'path': DIST_NPY, 'mode': 'divide', 'kernel': 'hill', 'd0': 8.0, 'n': 4, 'epsilon': 0.10})
print(f'  Hill(n=4)        -> {r["Hill(n=4)"]:.6f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
r['Surprise'] = run_one_batch('Surprise', {'path': DIST_NPY, 'mode': 'surprise', 'kernel': 'sigmoid', 'd0': 8.0, 'gamma': 1.5, 'epsilon': 0.10, 'alpha': 3.0})
print(f'  Surprise(a=3)    -> {r["Surprise"]:.6f}  ({time.time()-t0:.0f}s)')

b = r['Baseline']
print(f"\n{'='*60}")
print(f"  {'Method':<18s} {'Corr':>8s}  {'Delta':>8s}")
print(f"  {'-'*18} {'-'*8}  {'-'*8}")
for name, corr in r.items():
    d = corr - b
    flag = ' *' if d > 0 else ''
    print(f"  {name:<18s} {corr:8.4f}  {d:+8.4f}{flag}")
print(f"{'='*60}")